# NeuralScene Bench

Controlled novel-view synthesis benchmark for Netflix JR40251 portfolio work.

Initial scope: Mip-NeRF 360 Bonsai, comparing Nerfstudio `splatfacto` and `nerfacto` under a shared Nerfstudio evaluation pipeline.

Important: results from this notebook should be compared only within this benchmark. Do not directly mix these metrics with the earlier gsplat FastGS or SplatStream numbers because the training and evaluation pipelines differ.

Pinned Nerfstudio source revision: `50e0e3c70c775e89333256213363badbf074f29d`.


## Step 1. Check the Colab GPU environment
Run this cell first.


In [ ]:
import sys, subprocess
print('Python:', sys.version)
import torch
print('PyTorch:', torch.__version__)
print('CUDA build:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print('\n--- NVIDIA SMI ---')
subprocess.run(['nvidia-smi'])


## Step 2. Install the pinned benchmark environment

Creates an isolated Python 3.11 environment and pins Nerfstudio plus PyTorch/CUDA dependencies. Run once per fresh runtime.


In [ ]:
from pathlib import Path
import subprocess, sys
NS_COMMIT = '50e0e3c70c775e89333256213363badbf074f29d'
VENV = Path('/content/neuralscene-env')
NS = Path('/content/nerfstudio')
PY = VENV / 'bin/python'
def run(cmd, cwd=None):
    print('\n$', ' '.join(map(str, cmd)), flush=True)
    subprocess.run(list(map(str, cmd)), cwd=cwd, check=True)
run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
if not PY.exists(): run(['uv', 'venv', '--python', '3.11', VENV])
run(['uv', 'pip', 'install', '--python', PY, 'torch==2.7.1', 'torchvision==0.22.1', '--index-url', 'https://download.pytorch.org/whl/cu128'])
if not NS.exists(): run(['git', 'clone', 'https://github.com/nerfstudio-project/nerfstudio.git', NS])
run(['git', 'fetch', '--all', '--tags'], cwd=NS)
run(['git', 'reset', '--hard', NS_COMMIT], cwd=NS)
run(['uv', 'pip', 'install', '--python', PY, '-e', NS])
run([PY, '-c', "import torch; print('Environment ready'); print('Torch:', torch.__version__); print('CUDA:', torch.version.cuda); print('CUDA available:', torch.cuda.is_available()); print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)"])
print('\nPinned Nerfstudio commit:')
run(['git', 'rev-parse', 'HEAD'], cwd=NS)


## Step 3. Prepare Mip-NeRF 360 Bonsai

The NerfBaselines Bonsai mirror uses the factor-2 layout `images_2` plus `sparse_2/0`. The earlier notebook only looked for `sparse/0`, which caused the dataset-root detection error. This corrected cell recognizes both layouts and reuses the ZIP already downloaded in the current runtime.


In [ ]:
from pathlib import Path
import subprocess, zipfile, shutil
URL = 'https://data.ciirc.cvut.cz/public/projects/2023NerfBaselines/data/gaussian-splatting/mipnerf360/bonsai.zip'
ZIP = Path('/content/bonsai_mipnerf360.zip')
ROOT = Path('/content/data/mipnerf360')
SCENE = ROOT / 'bonsai'
ROOT.mkdir(parents=True, exist_ok=True)
def is_scene_root(p):
    return p.is_dir() and any((p/q).is_dir() for q in ['images','images_2','images_4']) and any((p/q).is_dir() for q in ['sparse/0','sparse_2/0','sparse_4/0','colmap/sparse/0'])
if not is_scene_root(SCENE):
    print('Preparing Bonsai. Reusing completed download bytes when available.', flush=True)
    subprocess.run(['wget', '-c', URL, '-O', str(ZIP)], check=True)
    if not zipfile.is_zipfile(ZIP): raise RuntimeError('Bonsai download is not a valid ZIP.')
    tmp = ROOT / '_bonsai_extract'
    shutil.rmtree(tmp, ignore_errors=True)
    tmp.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP, 'r') as zf:
        print('ZIP entries:', len(zf.infolist()))
        print('First ZIP entries:', zf.namelist()[:12])
        zf.extractall(tmp)
    candidates = ([tmp] if is_scene_root(tmp) else []) + [p for p in tmp.rglob('*') if is_scene_root(p)]
    if not candidates:
        dirs = sorted(str(p.relative_to(tmp)) for p in tmp.rglob('*') if p.is_dir())[:80]
        raise RuntimeError(f'Could not locate Bonsai root. Extracted directories: {dirs}')
    src = sorted(candidates, key=lambda p: len(p.parts))[0]
    print('Detected scene root:', src)
    if SCENE.exists(): shutil.rmtree(SCENE)
    if src == tmp:
        SCENE.mkdir(parents=True, exist_ok=True)
        for item in list(tmp.iterdir()): shutil.move(str(item), str(SCENE/item.name))
        shutil.rmtree(tmp, ignore_errors=True)
    else:
        shutil.move(str(src), str(SCENE))
        shutil.rmtree(tmp, ignore_errors=True)
image_dir = next((q for q in ['images_2','images','images_4'] if (SCENE/q).is_dir()), None)
colmap_dir = next((q for q in ['sparse_2/0','sparse/0','sparse_4/0','colmap/sparse/0'] if (SCENE/q).is_dir()), None)
print('\nBonsai root:', SCENE)
print('Image directory:', image_dir)
print('COLMAP directory:', colmap_dir)
print('Image count:', len(list((SCENE/image_dir).iterdir())) if image_dir else 0)
print('COLMAP files:', sorted(p.name for p in (SCENE/colmap_dir).iterdir()) if colmap_dir else [])
print('Top-level:', sorted(p.name for p in SCENE.iterdir()))
if not image_dir or not colmap_dir: raise RuntimeError('Expected Bonsai image/COLMAP layout is incomplete.')
print('\nBONSAI DATA READY')


## Next

After Step 3 ends with `BONSAI DATA READY`, the next notebook update will add a short Splatfacto smoke test before any full benchmark run.
